# Train and run inference on Google Colab
Select **Runtime → Change runtime type → T4 GPU** before running all cells. Checkpoints and tokenizers are copied to Google Drive.

In [ ]:
from pathlib import Path

path = Path("/content/transformer/train.py")
code = path.read_text()

code = code.replace(
    "load_dataset('opus_books',",
    "load_dataset('Helsinki-NLP/opus_books',"
)

path.write_text(code)
print("Dataset identifier patched.")

In [14]:
!grep -n "load_dataset" /content/transformer/train.py

16:from datasets import load_dataset
115:    ds_raw = load_dataset('Helsinki-NLP/opus_books', f'{config["src_lang"]}-{config["tgt_lang"]}', split='train')


In [15]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fri Sep  4 05:32:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

In [16]:
!git clone https://github.com/IshitaBhatt486/Transformer-from-scratch-using-PyTorch.git /content/transformer
%cd /content/transformer
!pip install -q -r requirements.txt

fatal: destination path '/content/transformer' already exists and is not an empty directory.
/content/transformer


## Train
The first run starts fresh. For later sessions, copy a Drive checkpoint back and add `--preload NN`.

In [17]:
import os

os.makedirs(
    "/content/drive/MyDrive/transformer-artifacts/weights",
    exist_ok=True
)

os.environ["MODEL_FOLDER"] = (
    "/content/drive/MyDrive/transformer-artifacts/weights"
)

In [18]:
from pathlib import Path

path = Path("/content/transformer/config.py")
code = path.read_text()

code = code.replace(
    '"model_folder": "weights"',
    '"model_folder": os.getenv("MODEL_FOLDER", "weights")'
)

path.write_text(code)
print("Drive checkpoint storage configured.")

Drive checkpoint storage configured.


In [19]:
!cp tokenizer_en.json tokenizer_fr.json '/content/drive/MyDrive/transformer-artifacts/'

cp: cannot stat 'tokenizer_en.json': No such file or directory
cp: cannot stat 'tokenizer_fr.json': No such file or directory


In [ ]:
%cd /content/transformer
!python train.py --epochs 20 --batch-size 16 --num-workers 2

/content/transformer
2026-09-04 05:32:07.769042: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Using device: cuda
README.md: 100% 28.1k/28.1k [00:00<00:00, 48.1MB/s]

en-fr/train-00000-of-00001.parquet: downloading bytes:  92% 19.4M/21.0M [00:01<00:00, 26.2MB/s,  897kB/s  ]
en-fr/train-00000-of-00001.parquet: downloading bytes: 100% 20.4M/20.4M [00:01<00:00, 18.4MB/s, 1.98MB/s  ]
en-fr/train-00000-of-00001.parquet: reconstructing file: 100% 21.0M/21.0M [00:01<00:00, 18.9MB/s, 2.05MB/s  ]
Generating train split: 100% 127085/127085 [00:00<00:00, 581300.86 examples/s]
[00:00:08] Pre-processing sequences       ██████████████████ 0        /        0Using 126,288/127,085 examples that fit seq_length=128
Max length of source sentences: 126
Max length 

In [ ]:
!cp tokenizer_en.json tokenizer_fr.json \
  "/content/drive/MyDrive/transformer-artifacts/"

In [ ]:
!find "/content/drive/MyDrive/transformer-artifacts" -maxdepth 2 -type f

## Inference

In [ ]:
!python translate.py 'This model is ready for inference.'

In [ ]:
!python translate.py "I would like to read a book."

# Optional API test

In [ ]:
import subprocess
import time
import requests

server = subprocess.Popen([
    "uvicorn", "app:app",
    "--host", "127.0.0.1",
    "--port", "8000"
])

for attempt in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )
        if response.ok:
            print("API ready:", response.json())
            break
    except requests.RequestException:
        time.sleep(2)
else:
    server.terminate()
    raise RuntimeError("API did not start; inspect the uvicorn output above.")

response = requests.post(
    "http://127.0.0.1:8000/translate",
    json={"text": "Hello world"},
    timeout=60
)

print(response.status_code, response.json())
server.terminate()